<a href="https://colab.research.google.com/github/roymukta620-byte/DataScience-Projects/blob/main/Syntactic_analysis_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [8]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy loaded successfully")


spaCy loaded successfully


In [3]:
sentence = "The couple ordered a pizza with pepperoni."

doc = nlp(sentence)

for token in doc:
    print(
        token.text,
        " | POS:",
        token.pos_,
        " | Dependency:",
        token.dep_,
        " | Head:",
        token.head.text
    )



The  | POS: DET  | Dependency: det  | Head: couple
couple  | POS: NOUN  | Dependency: nsubj  | Head: ordered
ordered  | POS: VERB  | Dependency: ROOT  | Head: ordered
a  | POS: DET  | Dependency: det  | Head: pizza
pizza  | POS: NOUN  | Dependency: dobj  | Head: ordered
with  | POS: ADP  | Dependency: prep  | Head: pizza
pepperoni  | POS: NOUN  | Dependency: pobj  | Head: with
.  | POS: PUNCT  | Dependency: punct  | Head: ordered


**Extract noun phrases**

In [4]:
for chunk in doc.noun_chunks:
    print(chunk.text)

The couple
a pizza
pepperoni


here in the above output:
        *

1.   The couple
2.   a pizza
3.    pepperoni





These are NP candidates.

**Step 6 — Extract prepositional phrases Now:**

In [5]:
for token in doc:
    if token.pos_ == "ADP":
        print("Preposition:", token.text)

        for child in token.children:
            print("  Child:", child.text)

Preposition: with
  Child: pepperoni


**Step 7 — First simple NP VP NP PP detector Now we create a function:**

In [6]:
def find_np_vp_np_pp(sentence):

    doc = nlp(sentence)

    has_subject = False
    has_verb = False
    has_object = False
    has_pp = False

    for token in doc:

        # subject NP
        if token.dep_ in ["nsubj"]:
            has_subject = True

        # verb
        if token.pos_ == "VERB":
            has_verb = True

        # object NP
        if token.dep_ in ["dobj", "obj"]:
            has_object = True

        # PP
        if token.pos_ == "ADP":
            has_pp = True


    return (
        has_subject
        and has_verb
        and has_object
        and has_pp
    )

**Test:**

In [7]:
sentence = "The couple ordered a pizza with pepperoni."

print(find_np_vp_np_pp(sentence))

True


In [9]:
def analyze_np_vp_np_pp(sentence):

    doc = nlp(sentence)

    result = {
        "Sentence": sentence,
        "Subject NP": None,
        "Verb": None,
        "Object NP": None,
        "PP": None
    }

    for token in doc:

        # Subject NP
        if token.dep_ == "nsubj":
            result["Subject NP"] = token.subtree
            result["Subject NP"] = " ".join(
                [child.text for child in token.subtree]
            )

        # Main Verb
        if token.pos_ == "VERB":
            result["Verb"] = token.text

        # Object NP
        if token.dep_ in ["dobj", "obj"]:
            result["Object NP"] = " ".join(
                [child.text for child in token.subtree]
            )

        # PP detection
        if token.pos_ == "ADP":
            pp_words = [token.text]

            for child in token.children:
                pp_words.append(child.text)

            result["PP"] = " ".join(pp_words)


    return result

In [11]:
sentence = "The man cut the bread with a knife."

analysis = analyze_np_vp_np_pp(sentence)

for key, value in analysis.items():
    print(key, ":", value)

Sentence : The man cut the bread with a knife.
Subject NP : The man
Verb : cut
Object NP : the bread
PP : with knife
